In [1]:
import pandas
import ast
from algorithms import *
import torch
import torch.nn as nn
import torch.nn.functional as functional
import torch.optim as optim
import transformers
from models import *
from transformers import AutoModel, AutoTokenizer
import wandb
from torch.utils.data import DataLoader
import os
import time
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import time
import json
from os import listdir
from os.path import isfile, join
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils
import json
import pathlib


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/jacopo/hlt/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(device)
print(torch.cuda.get_device_name(device))

cuda
NVIDIA GeForce RTX 4050 Laptop GPU


In [3]:
# class to create the DataSet object, as required by pytorch
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer):
        self.len = len(dataframe)

        # tokenized dataframe
        self.input_values = [tokenizer(a, padding="max_length", max_length=max_token_len,  return_tensors='pt', truncation=True) for a in dataframe["processed_tweet"].values]
        # gold labels
        self.labels = torch.from_numpy(dataframe[['NO', 'IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 
                                                  'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']].values.astype(np.float32))

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        input_values = self.input_values[idx]
        labels = self.labels[idx]

        return input_values, labels

In [4]:
def my_collate_fn(batch):
    
    # tokenized samples (inputs and targets) are grouped in batches
    input = {'input_ids':torch.stack(([x[0]['input_ids'][0] for x in batch])).to(device), 'attention_mask':torch.stack(([x[0]['attention_mask'][0] for x in batch])).to(device)}
    labels = torch.stack(([x[1] for x in batch])).to(device)
        
    return [input, labels]

In [5]:
def save_model(model:nn.Module, path:str):
    torch.save(model.state_dict(), path)

def load_model(path:str, hyperparam:dict): 
    if hyperparam['class'] == 'ff':
        classifier = MultiLabelClassifier(hyperparam['pretrained_name'], hyperparam['dropout'], 
                                          hyperparam['hidden_layer_1'], hyperparam['hidden_layer_2'], hyperparam['transf_out'])
    else:
        classifier = MultiLabelClassifierChain(hyperparam['pretrained_name'], hyperparam['dropout'], 
                                               hyperparam['hidden_layer_1'], hyperparam['hidden_layer_2'], hyperparam['transf_out'])
    
    classifier.load_state_dict(torch.load(path))

    return classifier

In [17]:
# class to create the DataSet object, as required by pytorch
class TestDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_token_len):
        self.len = len(dataframe)

        # tokenized dataframe
        self.input_values = [tokenizer(a, padding="max_length", max_length=max_token_len,  return_tensors='pt', truncation=True) for a in dataframe["processed_tweet"].values]

    def __len__(self):
        return self.len

    def __getitem__(self, idx):
        input_values = self.input_values[idx]

        return input_values

def test_collate_fn(batch):
    # tokenized samples (inputs and targets) are grouped in batches
    input = {'input_ids':torch.stack(([x['input_ids'][0] for x in batch])).to(device), 'attention_mask':torch.stack(([x['attention_mask'][0] for x in batch])).to(device)}
        
    return input

In [20]:
def model_json_output(model_path:str, hyperparam:dict, test_df_path:str, json_path:str = 'task3_soft_Medusa_1.json', categories:list = ['NO', 'IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE', 
                                                                                            'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']):
    model = load_model(model_path, hyperparam)
    model.to('cuda')
    tokenizer = AutoTokenizer.from_pretrained(hyperparam['pretrained_name'])
    test_df = pandas.read_csv(test_df_path, sep=";")

    ids = test_df['id_EXIST'].to_list() # ids necessary for the json format of the challenge
    test_data = TestDataset(test_df, tokenizer, hyperparam['max_token_length'])
    test_data_loader = DataLoader(test_data, batch_size=hyperparam['batch_size'], shuffle=False, collate_fn=test_collate_fn)

    json_data = [] # list of dict

    model.eval()
    with torch.no_grad():
        for ts_input in test_data_loader:
            tensor_output = model(ts_input)

            for output in tensor_output: # iterate over the batch output
                soft_pred = {}

                for i, category in enumerate(categories):
                    soft_pred[category] = output[i].item() # save soft pred in a dict (the categories are the keys)

                json_data.append({"test_case": "EXIST2024", "id": str(ids.pop(0)), "value": soft_pred})
    
    with open(json_path, 'w') as json_file: # save all dicts in the json file
        json.dump(json_data, json_file, indent=1)

In [12]:
model = MultiLabelClassifier('FacebookAI/xlm-roberta-base', 0.3, 10, 10, 768)
save_model(model, 'prova.pt')

config = {
    'pretrained_name' : 'FacebookAI/xlm-roberta-base',
    'max_token_length': 128,
    'batch_size' : 1,
    'dropout': 0.5,
    'hidden_layer_1': 10,
    'hidden_layer_2': 10,
    'transf_out': 768,
    'class': 'ff'
}

In [21]:
model_json_output('prova.pt', config, '../data/real_test_proc.csv')

In [22]:
evaluation_prediction('task3_soft_Medusa_1.json', '../data/evaluation/golds/EXIST2024_training_task3_gold_soft.json', metrics=['ICMSoft'])

2024-05-08 19:35:42,601 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft']
2024-05-08 19:35:43,796 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method
cargado 24


([-9.88392615726434], {})

In [24]:
evaluation_prediction('task3_soft_Medusa_1.json', '../data/evaluation/golds/EXIST2024_training_task3_gold_soft.json', metrics=['ICMSoft'])

2024-05-08 19:41:15,641 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICMSoft']
2024-05-08 19:41:15,660 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method


([6.689226018959818], {})

In [13]:
df = pandas.read_csv('../data/real_test.csv', sep=';')
df.head()

,id_EXIST,lang,tweet,split
0,500001,es,@Eurogamer_es Todo gamergate desde el desarrol...,TEST_ES
1,500002,es,"@ArCaNgEl__23 @Benzenazi Hombre, no es compara...",TEST_ES
2,500003,es,yo buscando las empresas metidas en el gamerga...,TEST_ES
3,500004,es,"@jordirico Primero fue internet, luego el game...",TEST_ES
4,500005,es,@AlonsoQuijano12 Yo estuve metido en el gamerg...,TEST_ES


In [ ]:
def evaluate_model(model:nn.Module, data:DataLoader, hard_metrics=['ICM', 'FMeasure'], soft_metrics=['ICMSoft'], categories=['NO', 'IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE',  
                                                  'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']):
    
    hard_results = []
    soft_results = []
    hard_gold_results = []
    soft_gold_results = []
    id_json = 0

    model.eval()
    with torch.no_grad():
        for batch in data:
            input, labels = batch

            model_output = torch.zeros(len(input), 6)

            for output, label in zip(model_output, labels):
                soft_pred = {}
                soft_gold = {}

                hard_pred = []
                hard_gold = []

                for i, category in enumerate(categories):
                    soft_pred[category] = output[i].item()
                    soft_gold[category] = label[i].item()


                hard_pred, hard_gold = compute_hardlabels(soft_pred, soft_gold)

                if hard_pred:
                    hard_results.append({"test_case": "EXIST2024", "id": str(id_json), "value": hard_pred})
                if hard_gold:
                    hard_gold_results.append({"test_case": "EXIST2024", "id": str(id_json), "value": hard_gold})

                soft_results.append({"test_case": "EXIST2024", "id": str(id_json), "value": soft_pred})
                soft_gold_results.append({"test_case": "EXIST2024", "id": str(id_json), "value": soft_gold})
                id_json += 1

    tmp_hard_gold = open("tmp_hard_gold.json", 'w')
    json.dump(hard_gold_results, tmp_hard_gold, indent=1) 
    tmp_hard_gold.close()

    tmp_hard = open("tmp_hard.json", 'w')
    json.dump(hard_results, tmp_hard, indent=1) 
    tmp_hard.close()

    tmp_soft_gold = open("tmp_soft_gold.json", 'w')
    json.dump(soft_gold_results, tmp_soft_gold, indent=1)
    tmp_soft_gold.close() 
    
    tmp_soft = open("tmp_soft.json", 'w')    
    json.dump(soft_results, tmp_soft, indent=1) 
    tmp_soft.close() 
    
    hard, soft = evaluation_prediction('tmp_hard.json', 'tmp_hard_gold.json', hard_metrics, True), evaluation_prediction('tmp_soft.json', 'tmp_soft_gold.json', soft_metrics, True)

    hard_dict = {}
    soft_dict = {}
    for i, metric in enumerate(hard_metrics):
        hard_dict[metric] = hard[0][i]

    for i, metric in enumerate(soft_metrics):
        soft_dict[metric] = soft[0][i]

    return hard_dict, hard[1], soft_dict 


In [6]:
df = pandas.read_csv('../data/real_test.csv', sep=';')
df.head()

,id_EXIST,lang,tweet,split
0,500001,es,@Eurogamer_es Todo gamergate desde el desarrol...,TEST_ES
1,500002,es,"@ArCaNgEl__23 @Benzenazi Hombre, no es compara...",TEST_ES
2,500003,es,yo buscando las empresas metidas en el gamerga...,TEST_ES
3,500004,es,"@jordirico Primero fue internet, luego el game...",TEST_ES
4,500005,es,@AlonsoQuijano12 Yo estuve metido en el gamerg...,TEST_ES


In [ ]:
def dataframe_to_json(df:pandas.DataFrame, mode='soft', json_path:str=''):
    trnc_df = df[['id', 'hard_label', 'NO', 'IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE',  
                                                  'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']] # take the necessary columns
    
    json_data = [] # data to write in the json file
    for i in range(len(trnc_df)): # TODO: find a way to do this more efficient, if possible
        data_id = int(trnc_df['id'].iloc[i]) # retrieve the id

        if mode == 'hard':
            if not pandas.isna(trnc_df['hard_label'].iloc[i]): # if hard label is NaN it doesn't need to be written in the file
                json_data.append({'id': data_id, 'test_case': 'EXIST2024', 'value': ast.literal_eval(trnc_df['hard_label'].iloc[i])})
        else: # for soft format value must be in a dict form
            value = trnc_df[['NO', 'IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE',  
                                                  'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']].iloc[i].to_dict()
            
            json_data.append({'id': data_id, 'test_case': 'EXIST2024', 'value': value})

    
    with open(json_path, 'w') as res_file: # save into file json
        json.dump(json_data, res_file, indent=1)

In [ ]:
def my_collate_fn(batch):
    
    # tokenized samples (inputs and targets) are grouped in batches
    input = {'input_ids':torch.stack(([x[0]['input_ids'][0] for x in batch])).to(device), 'attention_mask':torch.stack(([x[0]['attention_mask'][0] for x in batch])).to(device)}
    labels = torch.stack(([x[1] for x in batch])).to(device)
        
    return [input, labels]

In [ ]:
# correct tokenizer releted to the selected pretrained transformer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# training and validation sets are refactored as "CustomDataset" objects
tr_data = CustomDataset(training_set, tokenizer=tokenizer)
val_data = CustomDataset(validation_set, tokenizer=tokenizer)

batch_size = 256

In [ ]:
model = MultiLabelClassifierChain(model_name, 0.2, 20, 20)
model.to(device)

In [ ]:
val_data = CustomDataset(validation_set, tokenizer=tokenizer)
val_data_loader = DataLoader(val_data, batch_size=256, shuffle=True, collate_fn=my_collate_fn)

In [ ]:
def compute_hardlabels(soft_pred:dict, soft_gold:dict):
    hard_pred = []
    hard_gold = []

    categories=['IDEOLOGICAL-INEQUALITY', 'STEREOTYPING-DOMINANCE',  
                'OBJECTIFICATION', 'SEXUAL-VIOLENCE', 'MISOGYNY-NON-SEXUAL-VIOLENCE']

    sexism_soft_gold = [soft_gold[key] for key in categories]
    sexism_soft_pred = [soft_pred[key] for key in categories]

    if soft_pred['NO'] > 0.5: 
        hard_pred.append('NO')
    elif soft_pred['NO'] == 0.5 and 0.5 in sexism_soft_pred:
        hard_pred = []
    else:
        hard_pred = [key for key in categories if soft_pred[key] > (1/5)]

    if soft_gold['NO'] > 0.5: 
        hard_gold.append('NO')
    elif soft_gold['NO'] == 0.5 and 0.5 in sexism_soft_gold:
        hard_gold = []
    else:
        hard_gold = [key for key in categories if soft_gold[key] > (1/5)]

    return hard_pred, hard_gold

In [49]:
evaluation_prediction('tmp_hard_gold.json', 'tmp_hard_gold.json', ['ICMSoft'], verbose=True)

2024-05-02 11:05:31,958 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['ICM', 'FMeasure']
2024-05-02 11:05:32,017 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM evaluation method
2024-05-02 11:05:32,040 - pyevall.metrics.metrics - INFO -             evaluate() - Executing fmeasure evaluation method
This is a table PyEvALL report, so no warnings or errors are shown. Please, check the embedded report to check errors if any metric has the value "-" or is an empty value or table.
+----+--------------------+-------+------+
|    | files              |   ICM |   F1 |
|----+--------------------+-------+------|
|  0 | tmp_hard_gold.json |   1.5 |    1 |
+----+--------------------+-------+------+
+----+------------------------------+-------+------+
|    | files                        |   ICM |   F1 |
|----+------------------------------+-------+------|
|  0 | tmp_hard_gold.json_EXIST2024 |   1.5 |    1 |
+----+-----------------

([1.5, 1.0],
 {'F1_NO': 1.0,
  'F1_MISOGYNY-NON-SEXUAL-VIOLENCE': 1.0,
  'F1_IDEOLOGICAL-INEQUALITY': 1.0})